# ETL del archivo en crudo `cast.parquet`

## Librerías

In [1]:
import pandas as pd
import ast
import os
import gc

## Extracción

In [2]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/cast.parquet?raw=true"

cast = (
    pd.read_parquet(
        url, 
        engine='fastparquet'
        )
    )

In [3]:
cast

,cast,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...",11862
...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...",227506


Ver, mas en detalle, un dato de la columna 'cast'.

In [4]:
cast.iloc[0]['cast']

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

In [5]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   id      45476 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 710.7+ KB


Los datos estan completos

In [6]:
cast.isnull().sum()

cast    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay duplicados.

In [7]:
cast['id'].duplicated().any()

True

Se eliminan los duplicados.

In [8]:
cast.drop_duplicates(
    subset='id', 
    inplace=True
    )

In [9]:
cast['id'].duplicated().any()

False

### Renombrar el nombre de la columna 'id' por 'movie_id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con la tabla de las peliculas (movies.parquet).

In [10]:
cast.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

In [11]:
cast.columns

Index(['cast', 'movie_id'], dtype='object')

### Desanidar la columna 'cast' y unirla a la columna 'movie_id'

Se crea una lista de diccionarios a partir de los datos anidados de la columna 'cast'.

In [12]:
elenco_por_pelicula = [
    {
        **actores, 
        'movie_id': 
            row['movie_id']
            } 
    for _, row 
    in cast.iterrows() 
    for actores 
    in ast.literal_eval(
        row['cast']
        )
    ]

### Crear un dataframe con el resultado

In [13]:
cast = pd.DataFrame(
    elenco_por_pelicula
    )

In [14]:
del elenco_por_pelicula
gc.collect()

209

In [15]:
cast

,cast_id,character,credit_id,gender,id,name,order,profile_path,movie_id
0,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg,862
1,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg,862
2,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg,862
3,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg,862
4,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg,862
...,...,...,...,...,...,...,...,...,...
562039,2,,52fe4ea59251416c7515d7d5,2,544742,Iwan Mosschuchin,0,None,227506
562040,3,,52fe4ea59251416c7515d7d9,1,1090923,Nathalie Lissenko,1,None,227506
562041,4,,52fe4ea59251416c7515d7dd,2,1136422,Pavel Pavlov,2,None,227506
562042,5,,52fe4ea59251416c7515d7e1,0,1261758,Aleksandr Chabrov,3,None,227506


Primera fila.

In [16]:
cast.iloc[0]

cast_id                                       14
character                          Woody (voice)
credit_id               52fe4284c3a36847f8024f95
gender                                         2
id                                            31
name                                   Tom Hanks
order                                          0
profile_path    /pQFoyx7rp09CJTAb932F2g8Nlho.jpg
movie_id                                     862
Name: 0, dtype: object

### Verificar nulos

Solo la columna 'profile_path' tiene nulos.

In [17]:
cast.isnull().sum()

cast_id              0
character            0
credit_id            0
gender               0
id                   0
name                 0
order                0
profile_path    173678
movie_id             0
dtype: int64

### Renombrar el nombre de la columna 'id' por 'person_id'

Se hace esto para que se identifique rapidamente que es el id de la persona.

In [18]:
cast.rename(
    columns={
        'id': 'person_id'
        }, 
    inplace=True
    )

In [19]:
('id' not in cast.columns 
 and 'person_id' in cast.columns)

True

### Eliminar la columnas 'credit_id', 'order' y 'profile_path'

Se hace esto porque son innecesarias en el contexto de la visualizacion de las peliculas, por parte de los usuarios.

In [20]:
innecesarias = [
    'credit_id', 
    'order', 
    'profile_path'
]

In [21]:
cast.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [22]:
estan_eliminadas = True

for eliminada in innecesarias:
    if eliminada in cast.columns:
        estan_eliminadas = False
        break
    
estan_eliminadas

True

In [23]:
for columna in cast.columns:
    print(columna)

cast_id
character
gender
person_id
name
movie_id


### Se cambia el tipo de las columnas 'cast_id', 'gender', 'person_id' y 'movie_id' a str

Se hace esto porque los id son etiquetas y gender es un dato categorico. El resto de los datos tienen el tipo correcto que es str.

In [24]:
cast.dtypes

cast_id       int64
character    object
gender        int64
person_id     int64
name         object
movie_id      int64
dtype: object

In [25]:
cast['cast_id'] = (
    cast['cast_id']
    .astype(str)
    )
cast['gender'] = (
    cast['gender']
    .astype(str)
    )
cast['movie_id'] = (
    cast['movie_id']
    .astype(str)
    )
cast['person_id'] = (
    cast['person_id']
    .astype(str)
    )

### Última revisión

In [26]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   cast_id    562044 non-null  object
 1   character  562044 non-null  object
 2   gender     562044 non-null  object
 3   person_id  562044 non-null  object
 4   name       562044 non-null  object
 5   movie_id   562044 non-null  object
dtypes: object(6)
memory usage: 25.7+ MB


## Carga

In [27]:
directorio_actual = (
    os.getcwd()
    )
directorio_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [28]:
directorio_del_proyecto = (
    os.path.dirname(
        os.path.dirname(
            directorio_actual
            )
        )
    )
directorio_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [29]:
directorio_a_exportar = (
    os.path.join(
        directorio_del_proyecto, 
        'data', 
        'ETL', 
        'cast.parquet'
        )
    )
directorio_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\cast.parquet'

In [30]:
cast.to_parquet(
    directorio_a_exportar
    )

In [31]:
del cast
gc.collect()

60